# ASG Airlines: End-to-End Data Engineering & Operational Analytics Pipeline

### Project Overview
ASG Airlines operates domestic flight routes across key hubs in India. This notebook implements a scalable, end-to-end PySpark pipeline to ingest raw flight telemetry, resolve schema anomalies, clean cross-day midnight crossings, and construct feature-rich datasets ready for business intelligence in Power BI.

---

## 1. Environment Configuration & Path Setup
Defining workspace file pointers and source storage locations for raw flight operational telemetry.

In [0]:
file_path = "/Workspace/Users/keerthanamr2005@gmail.com/Drafts/UseCase - Airlines.csv"

print(file_path)

/Workspace/Users/keerthanamr2005@gmail.com/Drafts/UseCase - Airlines.csv


### 1.1 Ingesting Raw Flight Data
Reading the CSV export without schema inference to inspect raw delimiters, column alignments, and structural issues.

In [0]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("quote", '"')
    .option("escape", '"')
    .csv(file_path)
)
display(df_raw)

flight_id,airline,source,destination,departure_time,arrival_time,duration,_c7,_c8,_c9,_c10
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00,null,null,null,null
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00,null,null,null,null
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00,null,null,null,null
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00,null,null,null,null
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00,null,null,null,null
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00,null,null,null,null
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00,null,null,null,null
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00,null,null,null,null
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00,null,null,null,null
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00,null,null,null,null


### 1.2 Raw Schema & Volume Profiling
Inspecting row counts, column counts, and nullable string datatypes to diagnose trailing delimiters and missing values.

In [0]:
print("Number of rows:", df_raw.count())
print("Number of columns:", len(df_raw.columns))
print("Columns:")
print(df_raw.columns)
df_raw.printSchema()

Number of rows: 1020
Number of columns: 11
Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', '_c7', '_c8', '_c9', '_c10']
root
 |-- flight_id: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- source: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)



In [0]:
expected_columns = [
    "flight_id",
    "airline",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "duration"
]

df_raw = df_raw.select(*expected_columns)

print("Rows:", df_raw.count())
print("Columns:", len(df_raw.columns))

print(df_raw.columns)

display(df_raw.limit(10))

Rows: 1020
Columns: 7
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']


flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00


In [0]:
display(
    df_raw.limit(10)
)

flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00


### 1.3 Bronze Layer Ingestion
Persisting raw telemetry into the Bronze Delta table (`asgn_bronze`) to preserve raw audit history.

In [0]:
bronze_df = df_raw

print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))

display(bronze_df)

Bronze rows: 1020
Bronze columns: 7


flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00


In [0]:
bronze_df = df_raw

print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))

display(bronze_df)

Bronze rows: 1020
Bronze columns: 7


flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00


In [0]:
bronze_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("asgn_bronze")

In [0]:
display(spark.table("asgn_bronze"))

flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00


## 2. Data Quality Profiling & Anomaly Auditing

### 2.1 Null Value & Carrier Distribution Audit
Scanning columns for missingness and quantifying unassigned or unknown carrier rows.


In [0]:
from pyspark.sql import functions as F

null_summary = df_raw.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
])

display(null_summary)

flight_id,airline,source,destination,departure_time,arrival_time,duration
0,41,0,0,0,0,0


In [0]:
display(
    df_raw
    .groupBy("airline")
    .count()
    .orderBy(F.desc("count"))
)

airline,count
IndiGo,249
SpiceJet,240
Air India,236
Vistara,223
null,41
UNKNOWN,31


### 2.2 Telemetry Duplicate Analysis
Identifying repeated flight identifiers and measuring multi-row duplication frequency.

In [0]:
duplicate_flight_ids = (
    df_raw
    .groupBy("flight_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(duplicate_flight_ids)


flight_id,count
AI031,2
UK139,2
SJ146,2
AI070,2
SJ037,2
AI020,2
UK049,2
AI242,2
SJ118,2
UK160,2


In [0]:
duplicate_rows = (
    df_raw
    .groupBy(*df_raw.columns)
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_rows)

flight_id,airline,source,destination,departure_time,arrival_time,duration,count
AI242,UNKNOWN,BLR,CCU,16:41:42,17:43:42,01:02:00,2
AI031,Air India,DEL,MAA,13:05:42,16:11:42,03:06:00,2
SJ146,SpiceJet,HYD,BLR,22:37:42,00:33:42,01:56:00,2
UK163,Vistara,DEL,HYD,07:22:42,09:04:42,01:42:00,2
AI043,Air India,CCU,DEL,00:28:42,04:37:42,04:09:00,2
UK139,null,BLR,MAA,10:54:42,14:31:42,03:37:00,2
AI020,null,BOM,BLR,03:53:42,06:21:42,02:28:00,2
AI070,Air India,CCU,DEL,00:05:42,01:44:42,01:39:00,2
SJ037,SpiceJet,BLR,BOM,19:14:42,22:31:42,03:17:00,2
SJ118,SpiceJet,DEL,BOM,01:08:42,02:33:42,01:25:00,2


In [0]:
duplicate_count = duplicate_rows.count()

print("Number of duplicate groups:", duplicate_count)

Number of duplicate groups: 15


In [0]:
duplicate_records = (
    duplicate_rows
    .select(F.sum(F.col("count")).alias("duplicate_records"))
)

display(duplicate_records)

duplicate_records
30


In [0]:
duplicate_records = (
    duplicate_rows
    .select(F.sum(F.col("count")).alias("duplicate_records"))
)

display(duplicate_records)

duplicate_records
30


## 3. Silver Layer Transformation & Standardization

### 3.1 Deduplication & Unaligned Field Removal
Eliminating exact duplicate telemetry records and dropping unaligned trailing delimiters.

In [0]:
before_count = df_raw.count()

df_clean = df_raw.dropDuplicates()

after_count = df_clean.count()

print("Records before cleaning:", before_count)
print("Records after removing exact duplicates:", after_count)
print("Exact duplicates removed:", before_count - after_count)

Records before cleaning: 1020
Records after removing exact duplicates: 1005
Exact duplicates removed: 15


In [0]:
from pyspark.sql import functions as F

display(
    df_clean
    .groupBy("airline")
    .count()
    .orderBy(F.desc("count"))
)

airline,count
IndiGo,249
SpiceJet,236
Air India,233
Vistara,218
null,39
UNKNOWN,30


### 3.2 Carrier Name Imputation via IATA Prefix
Recovering missing and `UNKNOWN` airline names by parsing the 2-character flight code prefix (`6F` $\rightarrow$ IndiGo, `SJ` $\rightarrow$ SpiceJet, `AI` $\rightarrow$ Air India, `UK` $\rightarrow$ Vistara).

In [0]:
silver_df = df_clean.withColumn(
    "airline_clean",
    F.when(
        F.col("airline").isNull()
        | (F.trim(F.col("airline")) == "")
        | (F.upper(F.trim(F.col("airline"))) == "UNKNOWN"),
        F.lit("Unknown")
    ).otherwise(F.trim(F.col("airline")))
)

display(
    silver_df
    .groupBy("airline_clean")
    .count()
    .orderBy(F.desc("count"))
)

airline_clean,count
IndiGo,249
SpiceJet,236
Air India,233
Vistara,218
Unknown,69


In [0]:
display(
    silver_df
    .select("departure_time", "arrival_time", "duration")
    .limit(20)
)


departure_time,arrival_time,duration
23:35:42,01:23:42,01:48:00
23:05:42,01:24:42,02:19:00
22:46:42,03:13:42,04:27:00
22:00:42,23:50:42,01:50:00
21:58:42,23:51:42,01:53:00
16:18:42,18:54:42,02:36:00
15:32:42,16:49:42,01:17:00
14:47:42,16:26:42,01:39:00
13:10:42,17:06:42,03:56:00
11:58:42,15:03:42,03:05:00


### 3.3 Overnight Flight & Cross-Midnight Duration Calculation
Resolving cross-day flights where arrival time is earlier than departure time by computing true elapsed seconds across the midnight boundary.

In [0]:
silver_df = silver_df.withColumn(
    "departure_ts",
    F.to_timestamp(
        F.concat(F.lit("1970-01-01 "), F.trim(F.col("departure_time"))),
        "yyyy-MM-dd HH:mm:ss"
    )
)

silver_df = silver_df.withColumn(
    "arrival_ts_raw",
    F.to_timestamp(
        F.concat(F.lit("1970-01-01 "), F.trim(F.col("arrival_time"))),
        "yyyy-MM-dd HH:mm:ss"
    )
)

silver_df = silver_df.withColumn(
    "is_overnight",
    F.col("arrival_ts_raw") < F.col("departure_ts")
)

silver_df = silver_df.withColumn(
    "arrival_ts",
    F.when(
        F.col("is_overnight"),
        F.col("arrival_ts_raw") + F.expr("INTERVAL 1 DAY")
    ).otherwise(F.col("arrival_ts_raw"))
)

display(
    silver_df.select(
        "departure_time",
        "arrival_time",
        "departure_ts",
        "arrival_ts",
        "is_overnight"
    ).limit(20)
)

departure_time,arrival_time,departure_ts,arrival_ts,is_overnight
23:35:42,01:23:42,1970-01-01T23:35:42.000Z,1970-01-02T01:23:42.000Z,true
23:05:42,01:24:42,1970-01-01T23:05:42.000Z,1970-01-02T01:24:42.000Z,true
22:46:42,03:13:42,1970-01-01T22:46:42.000Z,1970-01-02T03:13:42.000Z,true
22:00:42,23:50:42,1970-01-01T22:00:42.000Z,1970-01-01T23:50:42.000Z,false
21:58:42,23:51:42,1970-01-01T21:58:42.000Z,1970-01-01T23:51:42.000Z,false
16:18:42,18:54:42,1970-01-01T16:18:42.000Z,1970-01-01T18:54:42.000Z,false
15:32:42,16:49:42,1970-01-01T15:32:42.000Z,1970-01-01T16:49:42.000Z,false
14:47:42,16:26:42,1970-01-01T14:47:42.000Z,1970-01-01T16:26:42.000Z,false
13:10:42,17:06:42,1970-01-01T13:10:42.000Z,1970-01-01T17:06:42.000Z,false
11:58:42,15:03:42,1970-01-01T11:58:42.000Z,1970-01-01T15:03:42.000Z,false


In [0]:
silver_df = silver_df.withColumn(
    "calculated_duration_minutes",
    F.round(
        (
            F.col("arrival_ts").cast("long")
            - F.col("departure_ts").cast("long")
        ) / 60,
        2
    )
)

display(
    silver_df.select(
        "departure_time",
        "arrival_time",
        "duration",
        "calculated_duration_minutes",
        "is_overnight"
    ).limit(20)
)

departure_time,arrival_time,duration,calculated_duration_minutes,is_overnight
23:35:42,01:23:42,01:48:00,108.0,true
23:05:42,01:24:42,02:19:00,139.0,true
22:46:42,03:13:42,04:27:00,267.0,true
22:00:42,23:50:42,01:50:00,110.0,false
21:58:42,23:51:42,01:53:00,113.0,false
16:18:42,18:54:42,02:36:00,156.0,false
15:32:42,16:49:42,01:17:00,77.0,false
14:47:42,16:26:42,01:39:00,99.0,false
13:10:42,17:06:42,03:56:00,236.0,false
11:58:42,15:03:42,03:05:00,185.0,false


### 3.4 Duration Mismatch & Corrupt Record Flagging
Extracting recorded flight duration strings, comparing them against computed timestamps, and flagging duration discrepancies.

In [0]:
duration_parts = F.split(F.trim(F.col("duration")), ":")

silver_df = silver_df.withColumn(
    "provided_duration_minutes",
    F.round(
        duration_parts.getItem(0).cast("double") * 60
        + duration_parts.getItem(1).cast("double")
        + duration_parts.getItem(2).cast("double") / 60,
        2
    )
)

silver_df = silver_df.withColumn(
    "duration_mismatch_flag",
    F.abs(
        F.col("provided_duration_minutes")
        - F.col("calculated_duration_minutes")
    ) > 0.01
)

display(
    silver_df
    .filter(
        F.col("duration").isNull()
        | (F.trim(F.col("duration")) == "")
        | (~F.trim(F.col("duration")).rlike(r"^\d{2}:\d{2}:\d{2}$"))
    )
    .select(
        "flight_id",
        "airline_clean",
        "departure_time",
        "arrival_time",
        "duration"
    )
)

flight_id,airline_clean,departure_time,arrival_time,duration
SJ192,SpiceJet,18:45:42,23:45:42,###############################################################################################################################################################################################################################################################


In [0]:
duration_parts = F.split(F.trim(F.col("duration")), ":")

silver_df = silver_df.withColumn(
    "provided_duration_minutes",
    F.when(
        F.trim(F.col("duration")).rlike(r"^\d{2}:\d{2}:\d{2}$"),
        duration_parts.getItem(0).cast("double") * 60
        + duration_parts.getItem(1).cast("double")
        + duration_parts.getItem(2).cast("double") / 60
    ).otherwise(None)
)

silver_df = silver_df.withColumn(
    "duration_mismatch_flag",
    F.when(
        F.col("provided_duration_minutes").isNull(),
        True
    ).otherwise(
        F.abs(
            F.col("provided_duration_minutes")
            - F.col("calculated_duration_minutes")
        ) > 0.01
    )
)

display(
    silver_df
    .select(
        "flight_id",
        "duration",
        "provided_duration_minutes",
        "calculated_duration_minutes",
        "duration_mismatch_flag"
    )
    .filter(F.col("duration_mismatch_flag") == True)
)

flight_id,duration,provided_duration_minutes,calculated_duration_minutes,duration_mismatch_flag
SJ192,###############################################################################################################################################################################################################################################################,null,300.0,true


In [0]:
duration_issue_count = (
    silver_df
    .filter(F.col("duration_mismatch_flag") == True)
    .count()
)

print("Duration issues:", duration_issue_count)

Duration issues: 1


### 3.5 Operational Feature Engineering
Deriving directional airport corridors (`source-destination`) and categorizing departure times into operational blocks.

In [0]:
silver_df = silver_df.withColumn(
    "route",
    F.concat(
        F.col("source"),
        F.lit("-"),
        F.col("destination")
    )
)

silver_df = silver_df.withColumn(
    "departure_hour",
    F.hour(F.col("departure_ts"))
)

silver_df = silver_df.withColumn(
    "arrival_hour",
    F.hour(F.col("arrival_ts"))
)

silver_df = silver_df.withColumn(
    "departure_period",
    F.when(F.col("departure_hour") < 6, "Night")
    .when(F.col("departure_hour") < 12, "Morning")
    .when(F.col("departure_hour") < 17, "Afternoon")
    .when(F.col("departure_hour") < 21, "Evening")
    .otherwise("Night")
)

display(
    silver_df.select(
        "flight_id",
        "airline_clean",
        "source",
        "destination",
        "route",
        "departure_hour",
        "arrival_hour",
        "departure_period",
        "calculated_duration_minutes",
        "is_overnight"
    ).limit(20)
)

flight_id,airline_clean,source,destination,route,departure_hour,arrival_hour,departure_period,calculated_duration_minutes,is_overnight
AI155,Air India,BOM,CCU,BOM-CCU,23,1,Night,108.0,true
SJ158,SpiceJet,DEL,HYD,DEL-HYD,23,1,Night,139.0,true
6F026,IndiGo,BOM,CCU,BOM-CCU,22,3,Night,267.0,true
UK021,Vistara,HYD,MAA,HYD-MAA,22,23,Night,110.0,false
AI060,Air India,DEL,HYD,DEL-HYD,21,23,Night,113.0,false
AI221,Air India,MAA,HYD,MAA-HYD,16,18,Afternoon,156.0,false
6F097,IndiGo,BLR,BOM,BLR-BOM,15,16,Afternoon,77.0,false
SJ154,SpiceJet,BLR,HYD,BLR-HYD,14,16,Afternoon,99.0,false
SJ117,SpiceJet,MAA,BLR,MAA-BLR,13,17,Afternoon,236.0,false
SJ218,SpiceJet,DEL,HYD,DEL-HYD,11,15,Morning,185.0,false


### 3.6 Silver Delta Lake Materialization
Selecting cleaned, typed columns and persisting the normalized dataset into `asgn_silver`.

In [0]:
display(
    silver_df.select(
        "flight_id",
        "airline_clean",
        "route",
        "departure_hour",
        "arrival_hour",
        "departure_period",
        "calculated_duration_minutes",
        "is_overnight"
    ).limit(20)
)

flight_id,airline_clean,route,departure_hour,arrival_hour,departure_period,calculated_duration_minutes,is_overnight
AI155,Air India,BOM-CCU,23,1,Night,108.0,true
SJ158,SpiceJet,DEL-HYD,23,1,Night,139.0,true
6F026,IndiGo,BOM-CCU,22,3,Night,267.0,true
UK021,Vistara,HYD-MAA,22,23,Night,110.0,false
AI060,Air India,DEL-HYD,21,23,Night,113.0,false
AI221,Air India,MAA-HYD,16,18,Afternoon,156.0,false
6F097,IndiGo,BLR-BOM,15,16,Afternoon,77.0,false
SJ154,SpiceJet,BLR-HYD,14,16,Afternoon,99.0,false
SJ117,SpiceJet,MAA-BLR,13,17,Afternoon,236.0,false
SJ218,SpiceJet,DEL-HYD,11,15,Morning,185.0,false


In [0]:
silver_final = silver_df.select(
    "flight_id",
    "airline",
    "airline_clean",
    "source",
    "destination",
    "route",
    "departure_time",
    "arrival_time",
    "departure_ts",
    "arrival_ts",
    "duration",
    "provided_duration_minutes",
    "calculated_duration_minutes",
    "duration_mismatch_flag",
    "is_overnight",
    "departure_hour",
    "arrival_hour",
    "departure_period"
)

print("Silver records:", silver_final.count())
print("Silver columns:", len(silver_final.columns))

display(silver_final.limit(20))

Silver records: 1005
Silver columns: 18


flight_id,airline,airline_clean,source,destination,route,departure_time,arrival_time,departure_ts,arrival_ts,duration,provided_duration_minutes,calculated_duration_minutes,duration_mismatch_flag,is_overnight,departure_hour,arrival_hour,departure_period
AI155,Air India,Air India,BOM,CCU,BOM-CCU,23:35:42,01:23:42,1970-01-01T23:35:42.000Z,1970-01-02T01:23:42.000Z,01:48:00,108.0,108.0,false,true,23,1,Night
SJ158,SpiceJet,SpiceJet,DEL,HYD,DEL-HYD,23:05:42,01:24:42,1970-01-01T23:05:42.000Z,1970-01-02T01:24:42.000Z,02:19:00,139.0,139.0,false,true,23,1,Night
6F026,IndiGo,IndiGo,BOM,CCU,BOM-CCU,22:46:42,03:13:42,1970-01-01T22:46:42.000Z,1970-01-02T03:13:42.000Z,04:27:00,267.0,267.0,false,true,22,3,Night
UK021,Vistara,Vistara,HYD,MAA,HYD-MAA,22:00:42,23:50:42,1970-01-01T22:00:42.000Z,1970-01-01T23:50:42.000Z,01:50:00,110.0,110.0,false,false,22,23,Night
AI060,Air India,Air India,DEL,HYD,DEL-HYD,21:58:42,23:51:42,1970-01-01T21:58:42.000Z,1970-01-01T23:51:42.000Z,01:53:00,113.0,113.0,false,false,21,23,Night
AI221,Air India,Air India,MAA,HYD,MAA-HYD,16:18:42,18:54:42,1970-01-01T16:18:42.000Z,1970-01-01T18:54:42.000Z,02:36:00,156.0,156.0,false,false,16,18,Afternoon
6F097,IndiGo,IndiGo,BLR,BOM,BLR-BOM,15:32:42,16:49:42,1970-01-01T15:32:42.000Z,1970-01-01T16:49:42.000Z,01:17:00,77.0,77.0,false,false,15,16,Afternoon
SJ154,SpiceJet,SpiceJet,BLR,HYD,BLR-HYD,14:47:42,16:26:42,1970-01-01T14:47:42.000Z,1970-01-01T16:26:42.000Z,01:39:00,99.0,99.0,false,false,14,16,Afternoon
SJ117,SpiceJet,SpiceJet,MAA,BLR,MAA-BLR,13:10:42,17:06:42,1970-01-01T13:10:42.000Z,1970-01-01T17:06:42.000Z,03:56:00,236.0,236.0,false,false,13,17,Afternoon
SJ218,SpiceJet,SpiceJet,DEL,HYD,DEL-HYD,11:58:42,15:03:42,1970-01-01T11:58:42.000Z,1970-01-01T15:03:42.000Z,03:05:00,185.0,185.0,false,false,11,15,Morning


In [0]:
silver_final.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("asgn_silver")

In [0]:
display(spark.table("asgn_silver"))

flight_id,airline,airline_clean,source,destination,route,departure_time,arrival_time,departure_ts,arrival_ts,duration,provided_duration_minutes,calculated_duration_minutes,duration_mismatch_flag,is_overnight,departure_hour,arrival_hour,departure_period
AI155,Air India,Air India,BOM,CCU,BOM-CCU,23:35:42,01:23:42,1970-01-01T23:35:42.000Z,1970-01-02T01:23:42.000Z,01:48:00,108.0,108.0,false,true,23,1,Night
SJ158,SpiceJet,SpiceJet,DEL,HYD,DEL-HYD,23:05:42,01:24:42,1970-01-01T23:05:42.000Z,1970-01-02T01:24:42.000Z,02:19:00,139.0,139.0,false,true,23,1,Night
6F026,IndiGo,IndiGo,BOM,CCU,BOM-CCU,22:46:42,03:13:42,1970-01-01T22:46:42.000Z,1970-01-02T03:13:42.000Z,04:27:00,267.0,267.0,false,true,22,3,Night
UK021,Vistara,Vistara,HYD,MAA,HYD-MAA,22:00:42,23:50:42,1970-01-01T22:00:42.000Z,1970-01-01T23:50:42.000Z,01:50:00,110.0,110.0,false,false,22,23,Night
AI060,Air India,Air India,DEL,HYD,DEL-HYD,21:58:42,23:51:42,1970-01-01T21:58:42.000Z,1970-01-01T23:51:42.000Z,01:53:00,113.0,113.0,false,false,21,23,Night
AI221,Air India,Air India,MAA,HYD,MAA-HYD,16:18:42,18:54:42,1970-01-01T16:18:42.000Z,1970-01-01T18:54:42.000Z,02:36:00,156.0,156.0,false,false,16,18,Afternoon
6F097,IndiGo,IndiGo,BLR,BOM,BLR-BOM,15:32:42,16:49:42,1970-01-01T15:32:42.000Z,1970-01-01T16:49:42.000Z,01:17:00,77.0,77.0,false,false,15,16,Afternoon
SJ154,SpiceJet,SpiceJet,BLR,HYD,BLR-HYD,14:47:42,16:26:42,1970-01-01T14:47:42.000Z,1970-01-01T16:26:42.000Z,01:39:00,99.0,99.0,false,false,14,16,Afternoon
SJ117,SpiceJet,SpiceJet,MAA,BLR,MAA-BLR,13:10:42,17:06:42,1970-01-01T13:10:42.000Z,1970-01-01T17:06:42.000Z,03:56:00,236.0,236.0,false,false,13,17,Afternoon
SJ218,SpiceJet,SpiceJet,DEL,HYD,DEL-HYD,11:58:42,15:03:42,1970-01-01T11:58:42.000Z,1970-01-01T15:03:42.000Z,03:05:00,185.0,185.0,false,false,11,15,Morning


In [0]:
from pyspark.sql import functions as F

gold_airline = (
    silver_final
    .groupBy("airline_clean")
    .agg(
        F.count("*").alias("total_flights"),

        F.round(
            F.avg("calculated_duration_minutes"), 2
        ).alias("avg_duration_minutes"),

        F.sum(
            F.when(F.col("is_overnight"), 1).otherwise(0)
        ).alias("overnight_flights"),

        F.sum(
            F.when(F.col("duration_mismatch_flag"), 1).otherwise(0)
        ).alias("duration_errors")
    )
    .withColumn(
        "overnight_percentage",
        F.round(
            F.col("overnight_flights")
            / F.col("total_flights") * 100,
            2
        )
    )
    .orderBy(F.desc("total_flights"))
)

display(gold_airline)

airline_clean,total_flights,avg_duration_minutes,overnight_flights,duration_errors,overnight_percentage
IndiGo,249,167.7,30,0,12.05
SpiceJet,236,164.19,27,1,11.44
Air India,233,163.15,30,0,12.88
Vistara,218,162.68,27,0,12.39
Unknown,69,166.07,8,0,11.59


In [0]:
gold_airline.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("asgn_gold_airline")

In [0]:
display(spark.table("asgn_gold_airline"))

airline_clean,total_flights,avg_duration_minutes,overnight_flights,duration_errors,overnight_percentage
IndiGo,249,167.7,30,0,12.05
SpiceJet,236,164.19,27,1,11.44
Air India,233,163.15,30,0,12.88
Vistara,218,162.68,27,0,12.39
Unknown,69,166.07,8,0,11.59


### 3.6 Silver Delta Lake Materialization
Selecting cleaned, typed columns and persisting the normalized dataset into `asgn_silver`.

In [0]:
gold_route = (
    silver_final
    .groupBy("route")
    .agg(
        F.count("*").alias("total_flights"),

        F.round(
            F.avg("calculated_duration_minutes"), 2
        ).alias("avg_duration_minutes"),

        F.sum(
            F.when(F.col("is_overnight"), 1).otherwise(0)
        ).alias("overnight_flights")
    )
    .withColumn(
        "overnight_percentage",
        F.round(
            F.col("overnight_flights")
            / F.col("total_flights") * 100,
            2
        )
    )
    .orderBy(F.desc("total_flights"))
)

display(gold_route)

route,total_flights,avg_duration_minutes,overnight_flights,overnight_percentage
BOM-CCU,90,169.51,16,17.78
CCU-DEL,72,153.61,11,15.28
MAA-BLR,65,172.82,8,12.31
BLR-BOM,60,147.73,7,11.67
HYD-MAA,57,152.81,5,8.77
DEL-HYD,54,174.76,6,11.11
HYD-DEL,42,185.36,9,21.43
BOM-DEL,39,153.82,3,7.69
CCU-BOM,33,163.97,1,3.03
DEL-BLR,29,170.03,3,10.34


### 4.2 Network Corridor Workload (asgn_gold_route)
Aggregating flight frequencies, multi-carrier coverage, and duration averages per city pair.

In [0]:
gold_route.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("asgn_gold_route")

In [0]:
display(spark.table("asgn_gold_route"))

route,total_flights,avg_duration_minutes,overnight_flights,overnight_percentage
BOM-CCU,90,169.51,16,17.78
CCU-DEL,72,153.61,11,15.28
MAA-BLR,65,172.82,8,12.31
BLR-BOM,60,147.73,7,11.67
HYD-MAA,57,152.81,5,8.77
DEL-HYD,54,174.76,6,11.11
HYD-DEL,42,185.36,9,21.43
BOM-DEL,39,153.82,3,7.69
CCU-BOM,33,163.97,1,3.03
DEL-BLR,29,170.03,3,10.34


### 4.3 Data Pipeline Quality Metrics (asgn_gold_quality)
Logging automated data governance metrics, clean record rates, and resolved anomaly counts.

In [0]:
total_records = silver_final.count()

unknown_airlines = (
    silver_final
    .filter(F.col("airline_clean") == "Unknown")
    .count()
)

overnight_flights = (
    silver_final
    .filter(F.col("is_overnight") == True)
    .count()
)

duration_errors = (
    silver_final
    .filter(F.col("duration_mismatch_flag") == True)
    .count()
)

quality_data = [
    ("Total Records", total_records),
    ("Unknown Airlines", unknown_airlines),
    ("Overnight Flights", overnight_flights),
    ("Duration Issues", duration_errors)
]

quality_df = spark.createDataFrame(
    quality_data,
    ["metric", "value"]
)

display(quality_df)

metric,value
Total Records,1005
Unknown Airlines,69
Overnight Flights,122
Duration Issues,1


In [0]:
quality_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("asgn_gold_quality")

In [0]:
print("Gold tables created:")
print("1. asgn_gold_airline")
print("2. asgn_gold_route")
print("3. asgn_gold_quality")

Gold tables created:
1. asgn_gold_airline
2. asgn_gold_route
3. asgn_gold_quality


### 4.4 Gold Analytics & Validation Queries
Executing Spark SQL queries on materialized Gold Delta tables to validate operational metrics.

In [0]:
%sql
SELECT
    airline_clean,
    total_flights,
    ROUND(avg_duration_minutes, 2) AS avg_duration_minutes,
    overnight_flights,
    ROUND(overnight_percentage, 2) AS overnight_percentage,
    duration_errors
FROM asgn_gold_airline
ORDER BY total_flights DESC;

airline_clean,total_flights,avg_duration_minutes,overnight_flights,overnight_percentage,duration_errors
IndiGo,249,167.7,30,12.05,0
SpiceJet,236,164.19,27,11.44,1
Air India,233,163.15,30,12.88,0
Vistara,218,162.68,27,12.39,0
Unknown,69,166.07,8,11.59,0


In [0]:
%sql
SELECT
    route,
    total_flights,
    ROUND(avg_duration_minutes, 2) AS avg_duration_minutes,
    overnight_flights,
    ROUND(overnight_percentage, 2) AS overnight_percentage
FROM asgn_gold_route
ORDER BY total_flights DESC
LIMIT 10;

route,total_flights,avg_duration_minutes,overnight_flights,overnight_percentage
BOM-CCU,90,169.51,16,17.78
CCU-DEL,72,153.61,11,15.28
MAA-BLR,65,172.82,8,12.31
BLR-BOM,60,147.73,7,11.67
HYD-MAA,57,152.81,5,8.77
DEL-HYD,54,174.76,6,11.11
HYD-DEL,42,185.36,9,21.43
BOM-DEL,39,153.82,3,7.69
CCU-BOM,33,163.97,1,3.03
DEL-BLR,29,170.03,3,10.34


In [0]:
%sql
SELECT
    route,
    total_flights,
    overnight_flights,
    ROUND(overnight_percentage, 2) AS overnight_percentage
FROM asgn_gold_route
WHERE overnight_flights > 0
ORDER BY overnight_percentage DESC;

route,total_flights,overnight_flights,overnight_percentage
HYD-DEL,42,9,21.43
BLR-CCU,21,4,19.05
MAA-BOM,21,4,19.05
BOM-MAA,27,5,18.52
DEL-BOM,28,5,17.86
BOM-CCU,90,16,17.78
DEL-MAA,23,4,17.39
BLR-HYD,19,3,15.79
BOM-HYD,26,4,15.38
CCU-DEL,72,11,15.28


In [0]:
%sql
SELECT
    route,
    total_flights,
    ROUND(avg_duration_minutes, 2) AS avg_duration_minutes,
    overnight_flights
FROM asgn_gold_route
ORDER BY avg_duration_minutes DESC
LIMIT 10;

route,total_flights,avg_duration_minutes,overnight_flights
BLR-MAA,16,187.31,2
HYD-DEL,42,185.36,9
BOM-MAA,27,183.41,5
BOM-HYD,26,180.85,4
DEL-MAA,23,178.78,4
DEL-BOM,28,178.11,5
MAA-BOM,21,176.76,4
BOM-BLR,23,176.52,1
DEL-HYD,54,174.76,6
MAA-BLR,65,172.82,8


In [0]:
%sql
SELECT
    metric,
    value
FROM asgn_gold_quality
ORDER BY
    CASE metric
        WHEN 'Total Records' THEN 1
        WHEN 'Unknown Airlines' THEN 2
        WHEN 'Overnight Flights' THEN 3
        WHEN 'Duration Issues' THEN 4
        ELSE 5
    END;

metric,value
Total Records,1005
Unknown Airlines,69
Overnight Flights,122
Duration Issues,1


In [0]:
%sql
SELECT
    airline_clean,
    COUNT(*) AS total_flights,
    ROUND(AVG(calculated_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(MAX(calculated_duration_minutes), 2) AS longest_flight_minutes,
    ROUND(MIN(calculated_duration_minutes), 2) AS shortest_flight_minutes
FROM asgn_silver
GROUP BY airline_clean
ORDER BY avg_duration_minutes DESC;

airline_clean,total_flights,avg_duration_minutes,longest_flight_minutes,shortest_flight_minutes
IndiGo,249,167.7,300.0,30.0
Unknown,69,166.07,291.0,33.0
SpiceJet,236,164.19,300.0,32.0
Air India,233,163.15,299.0,35.0
Vistara,218,162.68,300.0,30.0


In [0]:
%sql
SELECT
    route,
    total_flights,
    overnight_flights,
    ROUND(overnight_percentage, 2) AS overnight_percentage
FROM asgn_gold_route
WHERE total_flights >= 20
ORDER BY overnight_percentage DESC;

route,total_flights,overnight_flights,overnight_percentage
HYD-DEL,42,9,21.43
MAA-BOM,21,4,19.05
BLR-CCU,21,4,19.05
BOM-MAA,27,5,18.52
DEL-BOM,28,5,17.86
BOM-CCU,90,16,17.78
DEL-MAA,23,4,17.39
BOM-HYD,26,4,15.38
CCU-DEL,72,11,15.28
CCU-BLR,20,3,15.0


## 5. Advanced Operational Analytics & Anomaly Screening
Refining feature pipelines for deeper statistical anomaly detection and business intelligence handoff.

In [0]:
file_path = "/Workspace/Users/keerthanamr2005@gmail.com/Drafts/UseCase - Airlines.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("quote", '"')
    .option("escape", '"')
    .csv(file_path)
)
display(df_raw)

flight_id,airline,source,destination,departure_time,arrival_time,duration,_c7,_c8,_c9,_c10
SJ010,SpiceJet,CCU,MAA,23:38:42,02:32:42,02:54:00,null,null,null,null
AI155,Air India,BOM,CCU,23:35:42,01:23:42,01:48:00,null,null,null,null
UK094,Vistara,BOM,CCU,23:26:42,01:11:42,01:45:00,null,null,null,null
AI245,Air India,BOM,CCU,23:07:42,01:43:42,02:36:00,null,null,null,null
AI192,Air India,MAA,BOM,23:05:42,04:04:42,04:59:00,null,null,null,null
SJ158,SpiceJet,DEL,HYD,23:05:42,01:24:42,02:19:00,null,null,null,null
6F196,IndiGo,CCU,MAA,23:04:42,00:47:42,01:43:00,null,null,null,null
AI080,Air India,BOM,HYD,23:03:42,00:35:42,01:32:00,null,null,null,null
6F025,IndiGo,BLR,BOM,23:02:42,23:57:42,00:55:00,null,null,null,null
6F251,UNKNOWN,DEL,BLR,22:56:42,23:37:42,00:41:00,null,null,null,null


In [0]:
from pyspark.sql import functions as F
clean_cols = ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']
df_base = df_raw.select(*[c for c in clean_cols if c in df_raw.columns])
df_dedup = df_base.dropDuplicates()

print(f"Count before dedup: {df_raw.count()} | Count after dedup: {df_dedup.count()}")

Count before dedup: 1020 | Count after dedup: 1005


In [0]:
# Derive airline code from flight_id prefix (first 2 characters)
df_imputed = df_dedup.withColumn(
    "airline_code", F.substring(F.col("flight_id"), 1, 2)
).withColumn(
    "airline_clean",
    F.when(
        (F.col("airline").isNull()) | (F.col("airline") == "UNKNOWN") | (F.trim(F.col("airline")) == ""),
        F.when(F.col("airline_code") == "6F", "IndiGo")
         .when(F.col("airline_code") == "SJ", "SpiceJet")
         .when(F.col("airline_code") == "AI", "Air India")
         .when(F.col("airline_code") == "UK", "Vistara")
         .otherwise("Other")
    ).otherwise(F.col("airline"))
).drop("airline").withColumnRenamed("airline_clean", "airline")

display(df_imputed.groupBy("airline").count())

airline,count
Air India,255
SpiceJet,247
IndiGo,273
Vistara,230


In [0]:

def time_to_seconds(col_name):
    return (
        F.unix_timestamp(F.col(col_name), "HH:mm:ss") - 
        F.unix_timestamp(F.lit("00:00:00"), "HH:mm:ss")
    )

df_times = df_imputed.withColumn(
    "dep_sec", time_to_seconds("departure_time")
).withColumn(
    "arr_sec", time_to_seconds("arrival_time")
).withColumn(
    "calculated_duration_sec",
    F.when(F.col("arr_sec") >= F.col("dep_sec"), F.col("arr_sec") - F.col("dep_sec"))
     .otherwise((F.col("arr_sec") + 86400) - F.col("dep_sec"))
).withColumn(
    "duration_minutes", F.round(F.col("calculated_duration_sec") / 60, 2)
).withColumn(
    "duration_clean",
    F.concat_ws(
        ":",
        F.lpad(F.floor(F.col("calculated_duration_sec") / 3600).cast("string"), 2, "0"),
        F.lpad(F.floor((F.col("calculated_duration_sec") % 3600) / 60).cast("string"), 2, "0"),
        F.lpad((F.col("calculated_duration_sec") % 60).cast("string"), 2, "0")
    )
).drop("dep_sec", "arr_sec", "duration").withColumnRenamed("duration_clean", "duration")

display(df_times)

flight_id,source,destination,departure_time,arrival_time,airline_code,airline,calculated_duration_sec,duration_minutes,duration
AI155,BOM,CCU,23:35:42,01:23:42,AI,Air India,6480,108.0,01:48:00
SJ158,DEL,HYD,23:05:42,01:24:42,SJ,SpiceJet,8340,139.0,02:19:00
6F026,BOM,CCU,22:46:42,03:13:42,6F,IndiGo,16020,267.0,04:27:00
UK021,HYD,MAA,22:00:42,23:50:42,UK,Vistara,6600,110.0,01:50:00
AI060,DEL,HYD,21:58:42,23:51:42,AI,Air India,6780,113.0,01:53:00
AI221,MAA,HYD,16:18:42,18:54:42,AI,Air India,9360,156.0,02:36:00
6F097,BLR,BOM,15:32:42,16:49:42,6F,IndiGo,4620,77.0,01:17:00
SJ154,BLR,HYD,14:47:42,16:26:42,SJ,SpiceJet,5940,99.0,01:39:00
SJ117,MAA,BLR,13:10:42,17:06:42,SJ,SpiceJet,14160,236.0,03:56:00
SJ218,DEL,HYD,11:58:42,15:03:42,SJ,SpiceJet,11100,185.0,03:05:00


In [0]:
df_features = df_times.withColumn(
    "route", F.concat(F.col("source"), F.lit(" -> "), F.col("destination"))
).withColumn(
    "dep_hour", F.hour(F.to_timestamp(F.col("departure_time"), "HH:mm:ss"))
).withColumn(
    "time_of_day_bucket",
    F.when((F.col("dep_hour") >= 5) & (F.col("dep_hour") < 12), "Morning")
     .when((F.col("dep_hour") >= 12) & (F.col("dep_hour") < 17), "Afternoon")
     .when((F.col("dep_hour") >= 17) & (F.col("dep_hour") < 21), "Evening")
     .otherwise("Night")
)

display(df_features)

flight_id,source,destination,departure_time,arrival_time,airline_code,airline,calculated_duration_sec,duration_minutes,duration,route,dep_hour,time_of_day_bucket
AI155,BOM,CCU,23:35:42,01:23:42,AI,Air India,6480,108.0,01:48:00,BOM -> CCU,23,Night
SJ158,DEL,HYD,23:05:42,01:24:42,SJ,SpiceJet,8340,139.0,02:19:00,DEL -> HYD,23,Night
6F026,BOM,CCU,22:46:42,03:13:42,6F,IndiGo,16020,267.0,04:27:00,BOM -> CCU,22,Night
UK021,HYD,MAA,22:00:42,23:50:42,UK,Vistara,6600,110.0,01:50:00,HYD -> MAA,22,Night
AI060,DEL,HYD,21:58:42,23:51:42,AI,Air India,6780,113.0,01:53:00,DEL -> HYD,21,Night
AI221,MAA,HYD,16:18:42,18:54:42,AI,Air India,9360,156.0,02:36:00,MAA -> HYD,16,Afternoon
6F097,BLR,BOM,15:32:42,16:49:42,6F,IndiGo,4620,77.0,01:17:00,BLR -> BOM,15,Afternoon
SJ154,BLR,HYD,14:47:42,16:26:42,SJ,SpiceJet,5940,99.0,01:39:00,BLR -> HYD,14,Afternoon
SJ117,MAA,BLR,13:10:42,17:06:42,SJ,SpiceJet,14160,236.0,03:56:00,MAA -> BLR,13,Afternoon
SJ218,DEL,HYD,11:58:42,15:03:42,SJ,SpiceJet,11100,185.0,03:05:00,DEL -> HYD,11,Morning


### 5.1 Carrier Workload & Corridor Density Analysis
Computing operational throughput, minimum/maximum flying times, and carrier corridor distribution.

In [0]:

airline_summary = df_features.groupBy("airline").agg(
    F.count("flight_id").alias("total_flights"),
    F.round(F.avg("duration_minutes"), 2).alias("avg_duration_mins"),
    F.min("duration_minutes").alias("min_duration_mins"),
    F.max("duration_minutes").alias("max_duration_mins")
).orderBy(F.desc("total_flights"))

display(airline_summary)
route_summary = df_features.groupBy("route").agg(
    F.count("flight_id").alias("flight_frequency"),
    F.countDistinct("airline").alias("unique_carriers"),
    F.round(F.avg("duration_minutes"), 2).alias("avg_route_duration_mins")
).orderBy(F.desc("flight_frequency"))

display(route_summary)

airline,total_flights,avg_duration_mins,min_duration_mins,max_duration_mins
IndiGo,273,164.95,30.0,300.0
Air India,255,165.39,35.0,299.0
SpiceJet,247,163.79,32.0,300.0
Vistara,230,164.28,30.0,300.0


route,flight_frequency,unique_carriers,avg_route_duration_mins
BOM -> CCU,90,4,169.51
CCU -> DEL,72,4,153.61
MAA -> BLR,65,4,172.82
BLR -> BOM,60,4,147.73
HYD -> MAA,57,4,152.81
DEL -> HYD,54,4,174.76
HYD -> DEL,42,4,185.36
BOM -> DEL,39,4,153.82
CCU -> BOM,33,4,163.97
DEL -> BLR,29,4,170.03


### 5.2 Airport Hub Stress Index (Inflow vs. Outflow)
Measuring directional balance across hub stations to detect aircraft repositioning risks and terminal congestion.

In [0]:
outflow = df_features.groupBy("source").agg(F.count("flight_id").alias("departures"))
inflow = df_features.groupBy("destination").agg(F.count("flight_id").alias("arrivals"))
hub_metrics = (
    outflow.join(inflow, outflow["source"] == inflow["destination"], "outer")
    .select(
        F.coalesce(F.col("source"), F.col("destination")).alias("airport"),
        F.coalesce(F.col("departures"), F.lit(0)).alias("departures"),
        F.coalesce(F.col("arrivals"), F.lit(0)).alias("arrivals")
    )
    .withColumn("total_movements", F.col("departures") + F.col("arrivals"))
    .withColumn("net_flow", F.col("departures") - F.col("arrivals"))
    .orderBy(F.desc("total_movements"))
)

display(hub_metrics)

airport,departures,arrivals,total_movements,net_flow
BOM,205,169,374,36
DEL,160,198,358,-38
CCU,170,187,357,-17
HYD,178,141,319,37
MAA,157,147,304,10
BLR,135,163,298,-28


### 5.3 Route Duration Outlier Detection ($+1.5\sigma$)
Employing route-partitioned window functions to compute baseline durations ($\mu$) and standard deviations ($\sigma$), flagging flights exceeding $+1.5\sigma$ as delay risks.

In [0]:
from pyspark.sql import Window

route_window = Window.partitionBy("route")

df_anomalies = (
    df_features
    .withColumn("route_mean_duration", F.round(F.avg("duration_minutes").over(route_window), 2))
    .withColumn("route_std_duration", F.round(F.stddev("duration_minutes").over(route_window), 2))
    .withColumn(
        "is_duration_outlier",
        F.when(
            (F.col("route_std_duration") > 0) & 
            (F.col("duration_minutes") > (F.col("route_mean_duration") + 1.5 * F.col("route_std_duration"))),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

anomaly_summary = (
    df_anomalies
    .groupBy("airline", "route")
    .agg(
        F.count("flight_id").alias("total_scheduled"),
        F.sum("is_duration_outlier").alias("delay_risk_flights"),
        F.round((F.sum("is_duration_outlier") / F.count("flight_id")) * 100, 2).alias("outlier_rate_pct")
    )
    .filter(F.col("delay_risk_flights") > 0)
    .orderBy(F.desc("outlier_rate_pct"), F.desc("delay_risk_flights"))
)

display(anomaly_summary)

airline,route,total_scheduled,delay_risk_flights,outlier_rate_pct
Vistara,MAA -> DEL,6,2,33.33
SpiceJet,BLR -> CCU,3,1,33.33
Vistara,CCU -> MAA,3,1,33.33
SpiceJet,MAA -> HYD,3,1,33.33
IndiGo,BOM -> BLR,7,2,28.57
IndiGo,BLR -> DEL,4,1,25.0
SpiceJet,CCU -> HYD,4,1,25.0
Vistara,DEL -> CCU,4,1,25.0
Vistara,DEL -> MAA,4,1,25.0
SpiceJet,HYD -> BOM,4,1,25.0


### 5.4 Peak Schedule Matrix
Cross-tabulating airline operations against departure blocks (Morning, Afternoon, Evening, Night) to identify peak failure windows.

In [0]:

time_matrix = (
    df_features
    .groupBy("airline")
    .pivot("time_of_day_bucket", ["Morning", "Afternoon", "Evening", "Night"])
    .agg(F.count("flight_id"))
    .na.fill(0)
)

display(time_matrix)


airline,Morning,Afternoon,Evening,Night
Air India,67,71,39,78
SpiceJet,61,63,50,73
IndiGo,66,74,43,90
Vistara,52,59,40,79


### 5.5 Final Data Curation & Parquet Storage
Registering the enriched operational view and committing final gold analytics to Delta Lake.

In [0]:

df_anomalies.createOrReplaceTempView("curated_airline_analytics")
df_anomalies.write.format("delta").mode("overwrite").saveAsTable("gold_flight_analytics")

## 6. Power BI Data Export & Governance

### 6.1 Exporting Analysis-Ready Dataset for Power BI
Exporting the 1,005 cleaned, anomaly-flagged records to Workspace storage for dashboard ingestion.

In [0]:

df_powerbi = df_anomalies.select(
    "flight_id",
    "airline",
    "source",
    "destination",
    "route",
    "departure_time",
    "arrival_time",
    "duration",
    "duration_minutes",
    "dep_hour",
    "time_of_day_bucket",
    "route_mean_duration",
    "route_std_duration",
    "is_duration_outlier"
)
pdf_clean = df_powerbi.toPandas()
workspace_export_path = "/Workspace/Users/keerthanamr2005@gmail.com/Drafts/Airlines_Cleaned_PowerBI.csv"
pdf_clean.to_csv(workspace_export_path, index=False)

print(f"File successfully saved at: {workspace_export_path}")
print(f"Total Clean Records: {len(pdf_clean)}")

File successfully saved at: /Workspace/Users/keerthanamr2005@gmail.com/Drafts/Airlines_Cleaned_PowerBI.csv
Total Clean Records: 1005


In [0]:
from pyspark.sql import functions as F
sample_pii_data = [
    ("SJ010", "Keerthana Raman", "keerthana.r@example.com", "+91-9876543210", "Z1234567"),
    ("AI155", "Aditya Sharma", "aditya.sharma@domain.in", "+91-9123456780", "A9876543"),
    ("UK094", "Sneha Patel", "sneha.p@company.org", "+91-9988776655", "K4567890")
]

columns_pii = ["flight_id", "passenger_name", "email", "phone_number", "passport_no"]
df_passenger_raw = spark.createDataFrame(sample_pii_data, columns_pii)
SALT_SECRET = "ASG_Airlines_Secure_Salt_2026"
df_passenger_secure = (
    df_passenger_raw
    .withColumn(
        "hashed_passenger_id",
        F.sha2(F.concat_ws("::", F.col("passport_no"), F.lit(SALT_SECRET)), 256)
    )
    .withColumn(
        "masked_email",
        F.regexp_replace(
            F.col("email"),
            r"(?<=.)[^@](?=[^@]*?[^@]@)",
            "*"
        )
    )
    .withColumn(
        "masked_phone",
        F.concat(F.lit("*******"), F.substring(F.col("phone_number"), -4, 4))
    )
    .withColumn(
        "masked_passport",
        F.concat(
            F.substring(F.col("passport_no"), 1, 1),
            F.lit("****"),
            F.substring(F.col("passport_no"), -2, 2)
        )
    )
    .drop("passenger_name", "email", "phone_number", "passport_no")
)

display(df_passenger_secure)

flight_id,hashed_passenger_id,masked_email,masked_phone,masked_passport
SJ010,d2a569fc60947391a2945bb1028d15b438d06cad367ebbf4cc4a81a1ae65f328,k*********r@example.com,*******3210,Z****67
AI155,3a13494d8fc1ade11009c2f22b420c91eaced8c62e34cf232e16b473cc59300b,a***********a@domain.in,*******6780,A****43
UK094,353f91d29b402c63d14c25645aa9fce5ff499a38c4baa128867bef5f47e8ec82,s*****p@company.org,*******6655,K****90


In [0]:
%sql
CREATE OR REPLACE VIEW v_silver_restricted_operations AS
SELECT 
    flight_id,
    airline_clean,
    source,
    destination,
    route,
    calculated_duration_minutes,
    departure_period
FROM asgn_silver
WHERE 
    is_member('ASG_Flight_Operations_Admin')
    OR (is_member('BLR_Station_Managers') AND (source = 'BLR' OR destination = 'BLR'))
    OR (is_member('BOM_Station_Managers') AND (source = 'BOM' OR destination = 'BOM'))
    OR (is_member('DEL_Station_Managers') AND (source = 'DEL' OR destination = 'DEL'));

Summary & Engineering Findings
1. Data Cleaning & Pipeline Improvements
Data Structure Cleanup: Removed unnecessary empty columns and corrected formatting issues that occurred while importing the raw flight data.
Missing Airline Information: Filled in missing or "UNKNOWN" airline names by identifying airline codes from the flight_id. For example, 6F was mapped to IndiGo, SJ to SpiceJet, AI to Air India, and UK to Vistara.
Flight Duration Correction: Fixed incorrect duration values such as ### and handled flights that crossed midnight by calculating the actual time between departure and arrival.
Duplicate Removal: Checked the dataset for repeated flight records and removed exact duplicates to ensure the analysis was based on unique records.
2. Key Operational Insights
Major Airport Hubs: Flight activity is highly concentrated around major airports such as DEL, BOM, BLR, CCU, HYD, and MAA. Some busy routes may face higher turnaround pressure during peak departure periods.
Duration Variations: Identified flights whose scheduled duration was significantly higher than the normal duration for their respective routes. These variations may indicate inefficient scheduling, ground delays, or additional routing buffers.
Airline Workload: The distribution of flights varies across airlines and different time periods such as Morning, Afternoon, Evening, and Night, highlighting differences in how carriers manage their schedules.